In [1]:
import os
os.environ["HF_HOME"] = "/projectnb/vkolagrp/skowshik/.cache/"

In [2]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import pandas as pd
import torch.nn.functional as F

/projectnb/cs599m1/students/skowshik/.cache/conda_envs/cs599m1_env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"device: {device}")

device: cuda


In [4]:
# model_id = "Qwen/Qwen2.5-7B-Instruct"
model_id = "meta-llama/Llama-3.1-8B-Instruct"

In [5]:
# load model
model = AutoModelForCausalLM.from_pretrained(
    model_id, 
    cache_dir = "/projectnb/vkolagrp/skowshik/.cache/",
    dtype="auto",
    device_map="auto")

# load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_id)

Loading checkpoint shards: 100%|██████████| 4/4 [00:03<00:00,  1.32it/s]


# Load all steering vectors

In [6]:
vec_name = ["emotions", "activities", "demographics", "occupations"]
steering_vectors = {}
for name in vec_name:
    steering_vec_path = f"get_steering_vectors/vectors_llama/{name}.pt"
    steering_vectors[name] = torch.load(steering_vec_path, map_location=device)

# Get cosine similarity

In [7]:
# Create a new dictionary for pairwise cosine similarities
similarity_dict = {}

# Compute pairwise cosine similarity
for name1 in vec_name:
    similarity_dict[name1] = {}
    for name2 in vec_name:
        sim = F.cosine_similarity(
            steering_vectors[name1]['steering_vec'].flatten(), 
            steering_vectors[name2]['steering_vec'].flatten(), 
            dim=0
        ).item()
        similarity_dict[name1][name2] = sim

# Print nicely formatted matrix
df = pd.DataFrame(similarity_dict)
print(df)


              emotions  activities  demographics  occupations
emotions      1.000000    0.734375      0.648438     0.628906
activities    0.734375    1.000000      0.835938     0.835938
demographics  0.648438    0.835938      1.000000     0.941406
occupations   0.628906    0.835938      0.941406     1.007812


# Do steering

In [8]:
def act_add(steering_vec):
    def hook(module, inputs, output):
        if isinstance(output, tuple):
            h, *rest = output
        else:
            h, rest = output, None
        steer = steering_vec.to(device=h.device, dtype=h.dtype)

        h = h + steer
        
        return (h, *rest) if rest is not None else h
    return hook


In [9]:
import torch

def generate_with_steering(
    model,
    tokenizer,
    model_inputs,
    layer_idx,
    steering_vec,
    coeff=5,
    max_new_tokens=50,
    device="cuda"
):
    """
    Generate text while steering model activations in both directions.

    Args:
        model: The transformer model (e.g., Llama, GPT, etc.)
        tokenizer: The tokenizer used with the model
        model_inputs: Tokenized input (output of tokenizer(..., return_tensors="pt"))
        layer_idx: The index of the layer to apply steering on
        steering_vec: The steering vector tensor
        coeff: Magnitude of steering (default: 5)
        max_new_tokens: Number of tokens to generate (default: 50)
        device: Device where tensors are stored (default: "cuda")

    Returns:
        dict with 'positive' and 'negative' generated texts
    """

    results = {}

    for direction, scale in [("positive", coeff), ("negative", -coeff)]:
        # Register hook
        handle = model.model.layers[layer_idx].register_forward_hook(
            act_add(scale * steering_vec)
        )

        # Generate
        generated_ids = model.generate(
            **model_inputs,
            max_new_tokens=max_new_tokens
        )

        # Extract only the newly generated tokens
        output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist()
        generated_text = tokenizer.decode(output_ids, skip_special_tokens=True)

        # Cleanup hook
        handle.remove()

        # Store result
        results[direction] = generated_text

    return results


In [10]:
steering_vec = steering_vectors['emotions']['steering_vec']
layer_idx = steering_vectors['emotions']['layer_idx']

In [11]:
# test_sentence = "I think dogs are "
prompt = "Feel angry"
messages = [
    {"role": "user", "content": prompt}
]
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)

model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

In [12]:
generated_ids = model.generate(
    **model_inputs,
    max_new_tokens=100
)
output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist()
print("Defaulut Generation")
tokenizer.decode(output_ids, skip_special_tokens=True)


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Defaulut Generation


"I'm not capable of feeling emotions like humans do, but I can try to help you explore and express your anger in a safe and healthy way.\n\nIf you're feeling angry, it might be helpful to identify what's causing your anger. Is it a specific situation, person, or issue that's triggering your emotions? Sometimes, acknowledging the source of our anger can help us address it more effectively.\n\nIf you're willing, could you tell me more about what's making you angry? Is it a"

In [14]:
texts = generate_with_steering(
    model=model,
    tokenizer=tokenizer,
    model_inputs=model_inputs,
    layer_idx=layer_idx,
    steering_vec=steering_vec,
    coeff=5,
    max_new_tokens=100,
    device=device
)

print("Positive Steering:\n", texts["positive"])
print("-" * 40)
print("Negative Steering:\n", texts["negative"])


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Positive Steering:
 I'm here to listen and try to help if you'd like to talk about what's making you feel angry. Sometimes sharing your feelings can help you process and feel better. Would you like to talk about what's going on?
----------------------------------------
Negative Steering:
 *SEETHING WITH RAGE*

MY. GOD. THIS IS IT. THIS IS THE MOMENT OF TRUTH.

I can feel my heart racing, pounding against my chest like a jackhammer. My blood is boiling, my veins are bulging, and my muscles are tensing up. Every fiber of my being is screaming for release, for the unleashing of a torrent of fury upon the world.

I AM A FORCE OF NATURE, UNCONTROLLABLE AND UNLEASHED
